# 2. Export the classifier to ExecuTorch (CPU)
Run in a **fresh runtime**. After the install cell, restart the session (Runtime > Restart session), then run from the next cell.

Versions: torch 2.10.0+cpu, torchvision 0.25.0+cpu, torchaudio 2.10.0+cpu, torchao 0.16.0, executorch 1.1.0. The Android app uses executorch-android 1.1.0.

In [ ]:
!pip install -q "torch==2.10.0" "torchvision==0.25.0" "torchaudio==2.10.0" "torchao==0.16.0" "executorch==1.1.0" --extra-index-url https://download.pytorch.org/whl/cpu

In [ ]:
!pip list 2>/dev/null | grep -i -E "^(executorch|torch|torchvision|torchaudio|torchao) "

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, json, torch, torch.nn as nn
from PIL import Image
from torchvision import models, transforms
from executorch.runtime import Runtime
from executorch.exir import to_edge_transform_and_lower
from executorch.backends.xnnpack.partition.xnnpack_partitioner import XnnpackPartitioner

BASE = "/content/drive/MyDrive/Waste assistant"
PTH_PATH = f"{BASE}/waste_classifier.pth"
PTE_PATH = f"{BASE}/waste_classifier.pte"
CLASSES_PATH = f"{BASE}/classes.json"
IMAGE_PATH = f"{BASE}/test_ewaste.jpg"      # any test photo
for p in [PTH_PATH, CLASSES_PATH, IMAGE_PATH]:
    print(p, "EXISTS" if os.path.exists(p) else "MISSING")
classes = json.load(open(CLASSES_PATH)); print(classes)

## Load the trained PyTorch model and a test image (same preprocessing as the app)

In [ ]:
ev = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])
x = ev(Image.open(IMAGE_PATH).convert("RGB")).unsqueeze(0).contiguous()

model = models.mobilenet_v3_small(weights=None)
model.classifier[3] = nn.Linear(model.classifier[3].in_features, len(classes))
model.load_state_dict(torch.load(PTH_PATH, map_location="cpu"))
model.eval()

with torch.no_grad():
    original = model(x)
p1 = torch.softmax(original, dim=1)[0]
print("PyTorch:", classes[p1.argmax().item()], f"{p1.max().item()*100:.2f}%")

## Export with the XNNPACK delegate

In [ ]:
ep = torch.export.export(model, (x,))
edge = to_edge_transform_and_lower(ep, partitioner=[XnnpackPartitioner()])
et = edge.to_executorch()
with open(PTE_PATH, "wb") as f:
    f.write(et.buffer)
print("Saved", PTE_PATH)

## Check: the .pte must give the same answer as PyTorch

In [ ]:
method = Runtime.get().load_program(PTE_PATH).load_method("forward")
out = method.execute([x])[0]
p2 = torch.softmax(out, dim=1)[0]
print("ExecuTorch:", classes[p2.argmax().item()], f"{p2.max().item()*100:.2f}%")
print("Max logit difference:", (original - out).abs().max().item())
# The difference should be tiny (about 1e-5). If every class shows about 12.5%,
# the export versions do not match: re-check the versions above.